# Cross-Sectional Regime Quadrant Analysis

**횡단면 모멘텀 기반 국면 분석**

기존 Time-Series 방식(절대수익률 > 0)과 달리, 유니버스 내 상대 순위(중앙값 대비)로 국면을 판정합니다.

| | Fast Rank (하위) | Fast Rank (상위) |
|---|---|---|
| **Slow Rank (상위)** | Pullback → Correction | **Bullish** |
| **Slow Rank (하위)** | **Bearish** | Bounce → Recovery |

- **Bullish**: Slow 상위 & Fast 상위 (강세 지속)
- **Pullback**: Slow 상위 & Fast 하위, 첫 달 (눌림목)
- **Correction**: Slow 상위 & Fast 하위, 2+달 (조정)
- **Bounce**: Slow 하위 & Fast 상위, 첫 달 (데드캣)
- **Recovery**: Slow 하위 & Fast 상위, 2+달 (회복)
- **Bearish**: Slow 하위 & Fast 하위 (약세 지속)

> **핵심 차이**: TS는 절대수익률 부호로 판정 → 시장 전체가 Bullish/Bearish 될 수 있음.
> CS는 상대순위로 판정 → 항상 약 절반이 상위, 절반이 하위.

In [18]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from src.data import load_price_data
from src.config import TICKER_PRESETS, TICKER_DESCRIPTIONS, DEFAULT_START_DATE

In [19]:
FAST_WINDOW = 3     # Fast trend 윈도우 (월)
SLOW_WINDOW = 11    # Slow trend 윈도우 (월)
BEAR_FLOOR = 0.0    # 약세 국면 최소 비중

# 비중: Bullish=Pullback > Recovery > Correction > Bearish=Bounce
def make_weights(bear_floor):
    spread = 1.0 - bear_floor
    return {
        'Bullish': 1.0,
        'Pullback': 1.0,
        'Recovery': bear_floor + spread * 2/3,
        'Correction': bear_floor + spread * 1/3,
        'Bounce': bear_floor,
        'Bearish': bear_floor,
    }
REGIME_WEIGHTS = make_weights(BEAR_FLOOR)

In [20]:
tickers = TICKER_PRESETS["Alpha (Default)"]
prices, missing = load_price_data(tickers, start_date=DEFAULT_START_DATE)
print(f"Loaded: {len(prices.columns)} tickers, Missing: {missing}")
print(f"Date range: {prices.index[0].date()} ~ {prices.index[-1].date()}")
prices.tail(3)

Loaded: 35 tickers, Missing: []
Date range: 2000-01-03 ~ 2026-04-03


Ticker,SMH,IGV,XAR,XBI,XME,XOP,PAVE,ARKK,MGK,MGV,...,XLK,XLC,XLF,XLE,XLV,XLI,XLY,XLP,XLB,XLU
Date,,,,,,,,,,,,,,,,,,,,,
2026-04-01,391.970001,79.769997,259.950012,128.550003,109.900002,174.850006,51.689999,68.400002,371.730011,145.309998,...,134.910004,111.239998,49.439999,58.970001,147.729996,164.429993,109.800003,81.459999,50.459999,46.110001
2026-04-02,392.320007,80.339996,259.579987,128.960007,110.779999,177.720001,51.299999,68.559998,371.839996,145.470001,...,135.990005,111.699997,49.529999,59.250000,146.809998,163.770004,108.150002,81.889999,50.410000,46.340000
2026-04-03,392.320007,80.339996,259.579987,128.960007,110.779999,177.720001,51.299999,68.559998,371.839996,145.470001,...,135.990005,111.699997,49.529999,59.250000,146.809998,163.770004,108.150002,81.889999,50.410000,46.340000


In [21]:
# 월말 가격으로 리샘플
monthly_prices = prices.resample('BME').last()

# 수익률 계산 (설정된 윈도우 사용)
fast_ret = monthly_prices.pct_change(FAST_WINDOW)
slow_ret = monthly_prices.pct_change(SLOW_WINDOW)

# 6-regime 색상
REGIME_COLORS = {
    'Bullish': '#1e8449',
    'Pullback': '#27ae60',
    'Correction': '#5dade2',
    'Recovery': '#2980b9',
    'Bounce': '#e74c3c',
    'Bearish': '#c0392b',
}

ALL_REGIMES = list(REGIME_COLORS.keys())

def classify_regime_df(slow_ret, fast_ret):
    """Vectorized 6-regime classification using cross-sectional median."""
    # Step 1: 횡단면 중앙값 대비 상대 순위
    slow_median = slow_ret.median(axis=1)
    fast_median = fast_ret.median(axis=1)

    slow_above = slow_ret.gt(slow_median, axis=0)
    fast_above = fast_ret.gt(fast_median, axis=0)

    # 4-quadrant base
    base = pd.DataFrame(
        np.select(
            [slow_above & fast_above,
             slow_above & ~fast_above,
             ~slow_above & fast_above,
             ~slow_above & ~fast_above],
            ['Bullish', 'slow+fast-', 'slow-fast+', 'Bearish'],
            default=None,
        ),
        index=slow_ret.index,
        columns=slow_ret.columns,
    )
    # NaN 마스킹: 원본 수익률이 NaN인 곳은 국면도 NaN
    nan_mask = slow_ret.isna() | fast_ret.isna()
    base[nan_mask] = None

    # Step 2: duration split
    prev_base = base.shift(1)
    is_continuation = base == prev_base

    mask_sf = base == 'slow+fast-'
    regime = base.copy()
    regime[mask_sf & ~is_continuation] = 'Pullback'
    regime[mask_sf & is_continuation] = 'Correction'

    mask_fs = base == 'slow-fast+'
    regime[mask_fs & ~is_continuation] = 'Bounce'
    regime[mask_fs & is_continuation] = 'Recovery'

    return regime

# 각 티커별 국면 판정 (cross-sectional)
regime_df = classify_regime_df(slow_ret, fast_ret)
regime_df = regime_df.dropna(how='all')
print(f'Fast={FAST_WINDOW}M, Slow={SLOW_WINDOW}M (Cross-Sectional)')
print(f"Regime data: {regime_df.shape[0]} months x {regime_df.shape[1]} tickers")

# 국면별 분포
latest = regime_df.iloc[-1].dropna()
for r in ALL_REGIMES:
    print(f"  {r}: {(latest == r).sum()}")
regime_df.tail(3)

Fast=3M, Slow=11M (Cross-Sectional)
Regime data: 305 months x 35 tickers
  Bullish: 11
  Pullback: 3
  Correction: 3
  Recovery: 3
  Bounce: 3
  Bearish: 12


Ticker,SMH,IGV,XAR,XBI,XME,XOP,PAVE,ARKK,MGK,MGV,...,XLK,XLC,XLF,XLE,XLV,XLI,XLY,XLP,XLB,XLU
Date,,,,,,,,,,,,,,,,,,,,,
2026-02-27,Bullish,Bearish,Bullish,Pullback,Bullish,Recovery,Bullish,Correction,Bearish,Recovery,...,Correction,Bearish,Bearish,Recovery,Bearish,Bullish,Bearish,Recovery,Recovery,Bearish
2026-03-31,Bullish,Bearish,Bullish,Bullish,Bullish,Bullish,Bullish,Correction,Bearish,Bearish,...,Correction,Bearish,Bearish,Bullish,Bearish,Bullish,Bearish,Recovery,Recovery,Bounce
2026-04-30,Bullish,Bearish,Pullback,Bullish,Pullback,Bullish,Bullish,Correction,Bearish,Bounce,...,Bearish,Bearish,Bearish,Bullish,Bearish,Bounce,Bearish,Recovery,Bullish,Recovery


In [22]:
# 최신 시점 국면 현황
latest_date = regime_df.index[-1]
latest_regime = regime_df.loc[latest_date].dropna()
latest_fast = fast_ret.loc[latest_date].dropna()
latest_slow = slow_ret.loc[latest_date].dropna()

# 현재 국면 요약 테이블
summary = pd.DataFrame({
    'Ticker': latest_regime.index,
    'Description': [TICKER_DESCRIPTIONS.get(t, t) for t in latest_regime.index],
    f'Slow ({SLOW_WINDOW}M)': [f"{latest_slow.get(t, 0):.1%}" for t in latest_regime.index],
    f'Fast ({FAST_WINDOW}M)': [f"{latest_fast.get(t, 0):.1%}" for t in latest_regime.index],
    'Regime': latest_regime.values,
}).set_index('Ticker')

print(f"Date: {latest_date.date()} (Cross-Sectional)\n")
for regime_name in ALL_REGIMES:
    subset = summary[summary['Regime'] == regime_name]
    if len(subset) > 0:
        print(f"{'='*50}")
        print(f" {regime_name}: {len(subset)} tickers")
        print(f"{'='*50}")
        print(subset[['Description', f'Slow ({SLOW_WINDOW}M)', f'Fast ({FAST_WINDOW}M)']].to_string())
        print()

Date: 2026-04-30 (Cross-Sectional)

 Bullish: 11 tickers
                 Description Slow (11M) Fast (3M)
Ticker                                           
SMH           Semiconductors      64.1%     -2.8%
XBI                  Biotech      63.4%      3.4%
XOP            Oil & Gas E&P      52.2%     27.2%
PAVE          Infrastructure      24.7%      1.1%
SCHD                Dividend      21.3%      3.3%
GDX              Gold Miners      88.1%      0.4%
283580.KS      China CSI 300      33.9%     -0.4%
487230.KS  US AI Power-Infra      48.6%     13.0%
ICLN            Clean Energy      45.1%     -0.6%
XLE                   Energy      50.0%     16.8%
XLB                Materials      19.1%      2.8%

 Pullback: 3 tickers
                Description Slow (11M) Fast (3M)
Ticker                                          
XAR     Aerospace & Defense      36.2%     -4.9%
XME         Metals & Mining      85.1%     -6.6%
URA                 Uranium      60.1%    -11.1%

 Correction: 3 tickers
  

In [23]:
# 사분면 산점도: X = Slow Trend, Y = Fast Trend
# 6 regimes: Pullback/Correction 같은 사분면, Bounce/Recovery 같은 사분면

TARGET_DATE = None  # 특정 날짜를 보려면 '2024-06-30' 등 입력, None이면 최신

if TARGET_DATE:
    target_ts = pd.Timestamp(TARGET_DATE)
    valid_dates = regime_df.index[regime_df.index <= target_ts]
    if len(valid_dates) == 0:
        raise ValueError(f'{TARGET_DATE} 이전 데이터가 없습니다.')
    plot_date = valid_dates[-1]
else:
    plot_date = regime_df.index[-1]

plot_regime = regime_df.loc[plot_date].dropna()
plot_fast = fast_ret.loc[plot_date].dropna()
plot_slow = slow_ret.loc[plot_date].dropna()

slow_label = f'Slow ({SLOW_WINDOW}M Return)'
fast_label = f'Fast ({FAST_WINDOW}M Return)'

scatter_data = pd.DataFrame({
    slow_label: plot_slow,
    fast_label: plot_fast,
    'Regime': plot_regime,
}).dropna()
scatter_data['Description'] = [TICKER_DESCRIPTIONS.get(t, t) for t in scatter_data.index]

# 사분면별 텍스트 방향
TEXTPOS_MAP = {
    'Bullish': 'top right',
    'Pullback': 'bottom right',
    'Correction': 'bottom right',
    'Bounce': 'top left',
    'Recovery': 'top left',
    'Bearish': 'bottom left',
}

# 첫 달(Pullback/Bounce)은 diamond, 2+달 및 나머지는 circle
MARKER_SYMBOL = {
    'Bullish': 'circle',
    'Pullback': 'diamond',
    'Correction': 'circle',
    'Bounce': 'diamond',
    'Recovery': 'circle',
    'Bearish': 'circle',
}

fig = go.Figure()

for regime_name in ALL_REGIMES:
    color = REGIME_COLORS[regime_name]
    mask = scatter_data['Regime'] == regime_name
    subset = scatter_data[mask]
    fig.add_trace(go.Scatter(
        x=subset[slow_label] if len(subset) > 0 else [None],
        y=subset[fast_label] if len(subset) > 0 else [None],
        mode='markers+text',
        name=regime_name,
        text=subset['Description'] if len(subset) > 0 else [None],
        textposition=TEXTPOS_MAP[regime_name],
        textfont=dict(size=8),
        marker=dict(size=10, color=color, symbol=MARKER_SYMBOL[regime_name],
                     line=dict(width=1, color='white')),
        hovertemplate='%{text}<br>' + f'{SLOW_WINDOW}M: ' + '%{x:.1%}<br>' + f'{FAST_WINDOW}M: ' + '%{y:.1%}<extra></extra>',
    ))

# 축선
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.add_vline(x=0, line_dash='dash', line_color='gray', opacity=0.5)

# 사분면 라벨
x_range = max(abs(scatter_data[slow_label].min()), abs(scatter_data[slow_label].max()))
y_range = max(abs(scatter_data[fast_label].min()), abs(scatter_data[fast_label].max()))

annotations = [
    dict(x=x_range*0.7,  y=y_range*0.8,  text='Bullish', font=dict(size=28, color='rgba(30,132,73,0.5)'), showarrow=False),
    dict(x=-x_range*0.35, y=y_range*0.85, text='Recovery', font=dict(size=22, color='rgba(41,128,185,0.5)'), showarrow=False),
    dict(x=-x_range*0.85, y=y_range*0.35, text='Bounce', font=dict(size=22, color='rgba(231,76,60,0.5)'), showarrow=False),
    dict(x=x_range*0.85,  y=-y_range*0.35, text='Pullback', font=dict(size=22, color='rgba(39,174,96,0.5)'), showarrow=False),
    dict(x=x_range*0.35,  y=-y_range*0.85, text='Correction', font=dict(size=22, color='rgba(93,173,226,0.5)'), showarrow=False),
    dict(x=-x_range*0.7, y=-y_range*0.8, text='Bearish', font=dict(size=28, color='rgba(192,57,43,0.5)'), showarrow=False),
]

fig.update_layout(
    title=f'Cross-Sectional Regime Quadrant ({plot_date.date()})',
    xaxis_title=f'\u2190 Slow Trend \u2192',
    yaxis_title=f'\u2190 Fast Trend \u2192',
    xaxis=dict(tickformat='.0%', zeroline=True, title_font=dict(size=16)),
    yaxis=dict(tickformat='.0%', zeroline=True, title_font=dict(size=16)),
    annotations=annotations,
    template='plotly_white',
    height=700,
    width=700,
)
fig.show()
print(f'Date: {plot_date.date()}')

Date: 2026-04-30


In [24]:
# 국면을 숫자로 인코딩 (히트맵용)
regime_encoding = {
    'Bullish': 5,
    'Pullback': 4,
    'Correction': 3,
    'Recovery': 2,
    'Bounce': 1,
    'Bearish': 0,
}
regime_numeric = regime_df.replace(regime_encoding).apply(pd.to_numeric, errors='coerce')

# 최근 24개월만 표시
recent = regime_numeric.tail(60).T
recent.index = [TICKER_DESCRIPTIONS.get(t, t) for t in recent.index]
recent.columns = [d.strftime('%Y-%m') for d in regime_numeric.tail(60).index]

# 커스텀 컬러스케일 (6단계)
colorscale = [
    [0.0, '#c0392b'],    # Bearish (dark red)
    [0.2, '#e74c3c'],    # Bounce (light red)
    [0.4, '#2980b9'],    # Recovery (dark blue)
    [0.6, '#5dade2'],    # Correction (light blue)
    [0.8, '#27ae60'],    # Pullback (light green)
    [1.0, '#1e8449'],    # Bullish (dark green)
]

# 히트맵용 텍스트
text_matrix = regime_df.tail(60).T.copy()
text_matrix.index = recent.index
text_matrix.columns = recent.columns
text_short = text_matrix.replace({
    'Bullish': 'Bull',
    'Pullback': 'Pull',
    'Correction': 'Corr',
    'Bounce': 'Bnce',
    'Recovery': 'Recv',
    'Bearish': 'Bear',
})

fig = go.Figure(data=go.Heatmap(
    z=recent.values,
    x=recent.columns,
    y=recent.index,
    text=text_short.values,
    texttemplate='%{text}',
    textfont=dict(size=9),
    colorscale=colorscale,
    zmin=0, zmax=5,
    colorbar=dict(
        tickvals=[0, 1, 2, 3, 4, 5],
        ticktext=['Bearish', 'Bounce', 'Recovery', 'Correction', 'Pullback', 'Bullish'],
    ),
    hovertemplate='%{y}<br>%{x}<br>%{text}<extra></extra>',
))

fig.update_layout(
    title='CS Regime History Heatmap (Last 24 Months)',
    xaxis_title='Month',
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
    height=max(500, len(recent) * 28),
    width=1100,
)
fig.show()

In [25]:
# 국면별 익월 평균수익률
monthly_ret_1m = monthly_prices.pct_change(1).shift(-1)  # index T에 T→T+1 수익률

regime_returns = {}
for regime_name in ALL_REGIMES:
    mask = regime_df == regime_name
    # 국면에 해당하는 종목들의 익월 수익률만 추출 → 동일가중 평균
    regime_returns[regime_name] = monthly_ret_1m[mask].mean(axis=1)

regime_ret_df = pd.DataFrame(regime_returns).dropna(how='all')

# 요약 통계
# 전체 평균 (모든 국면 통합 동일가중)
all_ret = monthly_ret_1m[regime_df.notna()].mean(axis=1)

summary_stats = pd.DataFrame({
    'Mean (ann.)': regime_ret_df.mean() * 12,
    'Median (monthly)': regime_ret_df.median(),
    'Std (ann.)': regime_ret_df.std() * np.sqrt(12),
    'Sharpe': (regime_ret_df.mean() * 12) / (regime_ret_df.std() * np.sqrt(12)),
    'Obs': regime_ret_df.count(),
})
# All row
summary_stats.loc['All'] = [
    all_ret.mean() * 12,
    all_ret.median(),
    all_ret.std() * np.sqrt(12),
    (all_ret.mean() * 12) / (all_ret.std() * np.sqrt(12)),
    all_ret.count(),
]
print("=== 국면별 익월 수익률 통계 (Cross-Sectional) ===")
print(summary_stats.to_string(formatters={
    'Mean (ann.)': '{:.2%}'.format,
    'Median (monthly)': '{:.2%}'.format,
    'Std (ann.)': '{:.2%}'.format,
    'Sharpe': '{:.2f}'.format,
    'Obs': '{:.0f}'.format,
}))

# 박스플롯
fig = go.Figure()
for regime_name in ALL_REGIMES:
    color = REGIME_COLORS[regime_name]
    vals = regime_ret_df[regime_name].dropna()
    fig.add_trace(go.Box(
        y=vals, name=regime_name,
        marker_color=color, line_color=color,
        boxmean=True,
    ))

fig.update_layout(
    title='Next-Month Return by CS Regime (Equal-Weighted)',
    yaxis=dict(tickformat='.1%', title='Monthly Return'),
    template='plotly_white',
    height=500, width=700,
    showlegend=False,
)
fig.show()

# 누적수익률 시계열
fig2 = go.Figure()
for regime_name in ALL_REGIMES:
    color = REGIME_COLORS[regime_name]
    cumret = (1 + regime_ret_df[regime_name].fillna(0)).cumprod()
    fig2.add_trace(go.Scatter(
        x=cumret.index, y=cumret.values,
        name=regime_name, line=dict(color=color, width=2),
    ))

fig2.update_layout(
    title='Cumulative Return by CS Regime (Equal-Weighted)',
    yaxis_title='Cumulative Return',
    template='plotly_white',
    height=400, width=900,
)
fig2.show()

=== 국면별 익월 수익률 통계 (Cross-Sectional) ===
           Mean (ann.) Median (monthly) Std (ann.) Sharpe Obs
Bullish         10.42%            1.21%     15.91%   0.65 304
Pullback        12.72%            1.57%     20.69%   0.61 267
Correction      14.98%            1.33%     20.29%   0.74 254
Recovery         7.13%            1.03%     19.12%   0.37 245
Bounce          10.66%            1.01%     21.68%   0.49 272
Bearish          8.03%            0.92%     19.59%   0.41 304
All              9.95%            1.28%     16.39%   0.61 304


In [26]:
# 국면 그룹핑: Bullish+Pullback (강세권), Correction, Recovery, Bearish+Bounce (약세권)
GROUPED_REGIMES = {
    'Bullish+Pullback': ['Bullish', 'Pullback'],
    'Correction': ['Correction'],
    'Recovery': ['Recovery'],
    'Bearish+Bounce': ['Bearish', 'Bounce'],
}

GROUPED_COLORS = {
    'Bullish+Pullback': '#27ae60',
    'Correction': '#5dade2',
    'Recovery': '#2980b9',
    'Bearish+Bounce': '#e74c3c',
}

grouped_returns = {}
for group_name, regimes in GROUPED_REGIMES.items():
    mask = regime_df.isin(regimes)
    grouped_returns[group_name] = monthly_ret_1m[mask].mean(axis=1)

grouped_ret_df = pd.DataFrame(grouped_returns).dropna(how='all')

# 요약 통계
grouped_stats = pd.DataFrame({
    'Mean (ann.)': grouped_ret_df.mean() * 12,
    'Median (monthly)': grouped_ret_df.median(),
    'Std (ann.)': grouped_ret_df.std() * np.sqrt(12),
    'Sharpe': (grouped_ret_df.mean() * 12) / (grouped_ret_df.std() * np.sqrt(12)),
    'Obs': grouped_ret_df.count(),
})
grouped_stats.loc['All'] = [
    all_ret.mean() * 12,
    all_ret.median(),
    all_ret.std() * np.sqrt(12),
    (all_ret.mean() * 12) / (all_ret.std() * np.sqrt(12)),
    all_ret.count(),
]
print("=== 국면 그룹별 익월 수익률 통계 (Cross-Sectional) ===")
print(grouped_stats.to_string(formatters={
    'Mean (ann.)': '{:.2%}'.format,
    'Median (monthly)': '{:.2%}'.format,
    'Std (ann.)': '{:.2%}'.format,
    'Sharpe': '{:.2f}'.format,
    'Obs': '{:.0f}'.format,
}))

# 박스플롯
fig = go.Figure()
for group_name, color in GROUPED_COLORS.items():
    vals = grouped_ret_df[group_name].dropna()
    fig.add_trace(go.Box(
        y=vals, name=group_name,
        marker_color=color, line_color=color,
        boxmean=True,
    ))

fig.update_layout(
    title='Next-Month Return by CS Regime Group (Equal-Weighted)',
    yaxis=dict(tickformat='.1%', title='Monthly Return'),
    template='plotly_white',
    height=500, width=700,
    showlegend=False,
)
fig.show()

=== 국면 그룹별 익월 수익률 통계 (Cross-Sectional) ===
                 Mean (ann.) Median (monthly) Std (ann.) Sharpe Obs
Bullish+Pullback      11.33%            1.35%     15.89%   0.71 304
Correction            14.98%            1.33%     20.29%   0.74 254
Recovery               7.13%            1.03%     19.12%   0.37 245
Bearish+Bounce         7.91%            0.90%     19.11%   0.41 304
All                    9.95%            1.28%     16.39%   0.61 304


In [27]:
# 그룹핑 사분면 산점도: Bullish+Pullback, Correction, Recovery, Bearish+Bounce
plot_regime_grouped = plot_regime.copy()
for ticker in plot_regime_grouped.index:
    r = plot_regime_grouped[ticker]
    if r in ('Bullish', 'Pullback'):
        plot_regime_grouped[ticker] = 'Bullish+Pullback'
    elif r in ('Bearish', 'Bounce'):
        plot_regime_grouped[ticker] = 'Bearish+Bounce'

scatter_grouped = pd.DataFrame({
    slow_label: plot_slow,
    fast_label: plot_fast,
    'Regime': plot_regime_grouped,
}).dropna()
scatter_grouped['Description'] = [TICKER_DESCRIPTIONS.get(t, t) for t in scatter_grouped.index]

GROUPED_TEXTPOS = {
    'Bullish+Pullback': 'top right',
    'Correction': 'bottom right',
    'Recovery': 'top left',
    'Bearish+Bounce': 'bottom left',
}

fig = go.Figure()

for group_name, color in GROUPED_COLORS.items():
    mask = scatter_grouped['Regime'] == group_name
    subset = scatter_grouped[mask]
    fig.add_trace(go.Scatter(
        x=subset[slow_label] if len(subset) > 0 else [None],
        y=subset[fast_label] if len(subset) > 0 else [None],
        mode='markers+text',
        name=group_name,
        text=subset['Description'] if len(subset) > 0 else [None],
        textposition=GROUPED_TEXTPOS[group_name],
        textfont=dict(size=8),
        marker=dict(size=10, color=color, line=dict(width=1, color='white')),
        hovertemplate='%{text}<br>' + f'{SLOW_WINDOW}M: ' + '%{x:.1%}<br>' + f'{FAST_WINDOW}M: ' + '%{y:.1%}<extra></extra>',
    ))

fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.add_vline(x=0, line_dash='dash', line_color='gray', opacity=0.5)

x_range = max(abs(scatter_grouped[slow_label].min()), abs(scatter_grouped[slow_label].max()))
y_range = max(abs(scatter_grouped[fast_label].min()), abs(scatter_grouped[fast_label].max()))

annotations = [
    dict(x=x_range*0.7,  y=y_range*0.8,  text='Bullish+Pullback', font=dict(size=22, color='rgba(46,204,113,0.5)'), showarrow=False),
    dict(x=-x_range*0.7, y=y_range*0.8,  text='Recovery', font=dict(size=22, color='rgba(41,128,185,0.5)'), showarrow=False),
    dict(x=x_range*0.7,  y=-y_range*0.8, text='Correction', font=dict(size=22, color='rgba(230,126,34,0.5)'), showarrow=False),
    dict(x=-x_range*0.7, y=-y_range*0.8, text='Bearish+Bounce', font=dict(size=22, color='rgba(231,76,60,0.5)'), showarrow=False),
]

fig.update_layout(
    title=f'CS Regime Group Quadrant ({plot_date.date()})',
    xaxis_title='\u2190 Slow Trend \u2192',
    yaxis_title='\u2190 Fast Trend \u2192',
    xaxis=dict(tickformat='.0%', zeroline=True, title_font=dict(size=16)),
    yaxis=dict(tickformat='.0%', zeroline=True, title_font=dict(size=16)),
    annotations=annotations,
    template='plotly_white',
    height=700,
    width=700,
)
fig.show()

In [28]:
def regime_backtest(prices, regime_df, regime_weights, tcost=0.002, backtest_start='2015-01-01'):
    """
    국면 기반 비중 차등 전략 백테스트.
    
    월말 국면 판단 → 다음달 비중 적용 (implementation lag).
    비중은 국면별 raw weight를 정규화하여 합이 1이 되도록 함.
    투자할 티커가 없으면(모두 약세) 현금 보유.
    """
    monthly_prices = prices.resample('BME').last()
    monthly_ret = monthly_prices.pct_change(1).shift(-1)  # 다음 달 수익률
    
    # 국면 기반 raw weight
    raw_weights = regime_df.replace(regime_weights)
    raw_weights = raw_weights.apply(pd.to_numeric, errors='coerce')
    
    # 정규화: 합이 1이 되도록
    weight_sum = raw_weights.sum(axis=1)
    weights = raw_weights.div(weight_sum, axis=0).fillna(0)
    
    # backtest 시작일 필터
    mask = weights.index >= pd.Timestamp(backtest_start)
    weights = weights.loc[mask]
    monthly_ret = monthly_ret.reindex(weights.index)
    
    # 공통 컬럼
    common_cols = weights.columns.intersection(monthly_ret.columns)
    weights = weights[common_cols]
    monthly_ret = monthly_ret[common_cols].fillna(0)
    
    # 거래비용: 비중 변화의 절대값 합 * tcost
    weight_diff = weights.diff().abs().sum(axis=1).fillna(0)
    
    # 포트폴리오 수익률
    port_ret = (weights * monthly_ret).sum(axis=1) - weight_diff * tcost
    
    # NAV
    nav = (1 + port_ret).cumprod()
    nav = nav.dropna()
    
    return nav, weights, port_ret


# 벤치마크: 동일가중 (국면 무관)
def equal_weight_backtest(prices, regime_df, tcost=0.002, backtest_start='2015-01-01'):
    """동일가중 벤치마크."""
    monthly_prices = prices.resample('BME').last()
    monthly_ret = monthly_prices.pct_change(1).shift(-1)
    
    # 데이터 있는 티커에 동일 비중
    available = regime_df.notna().astype(float)
    n_available = available.sum(axis=1)
    weights = available.div(n_available, axis=0).fillna(0)
    
    mask = weights.index >= pd.Timestamp(backtest_start)
    weights = weights.loc[mask]
    monthly_ret = monthly_ret.reindex(weights.index)
    
    common_cols = weights.columns.intersection(monthly_ret.columns)
    weights = weights[common_cols]
    monthly_ret = monthly_ret[common_cols].fillna(0)
    
    weight_diff = weights.diff().abs().sum(axis=1).fillna(0)
    port_ret = (weights * monthly_ret).sum(axis=1) - weight_diff * tcost
    
    nav = (1 + port_ret).cumprod().dropna()
    return nav, port_ret


# 실행 (상단 config의 REGIME_WEIGHTS 사용)
nav_regime, weights_regime, ret_regime = regime_backtest(prices, regime_df, REGIME_WEIGHTS)
nav_ew, ret_ew = equal_weight_backtest(prices, regime_df)

print(f"Backtest period: {nav_regime.index[0].date()} ~ {nav_regime.index[-1].date()}")
print(f"Regime weights: {REGIME_WEIGHTS}")

Backtest period: 2015-01-30 ~ 2026-04-30
Regime weights: {'Bullish': 1.0, 'Pullback': 1.0, 'Recovery': 0.6666666666666666, 'Correction': 0.3333333333333333, 'Bounce': 0.0, 'Bearish': 0.0}


In [29]:
def calc_stats(nav, ret, name):
    """성과 지표 계산."""
    years = (nav.index[-1] - nav.index[0]).days / 365.25
    cagr = (nav.iloc[-1] / nav.iloc[0]) ** (1 / years) - 1
    vol = ret.std() * np.sqrt(12)  # 월간 → 연율
    sharpe = cagr / vol if vol > 0 else 0
    
    # MDD
    cummax = nav.cummax()
    dd = (nav - cummax) / cummax
    mdd = dd.min()
    
    return {
        'Strategy': name,
        'CAGR': f"{cagr:.1%}",
        'Volatility': f"{vol:.1%}",
        'Sharpe': f"{sharpe:.2f}",
        'MDD': f"{mdd:.1%}",
        'Final NAV': f"{nav.iloc[-1]:.2f}",
    }

# 연간 회전율 계산
turnover_regime = weights_regime.diff().abs().sum(axis=1).mean() * 12
turnover_ew = equal_weight_backtest  # EW는 weights 미반환 — 별도 계산
# EW 턴오버: regime_df 기반 동일가중
available = regime_df.notna().astype(float)
n_available = available.sum(axis=1)
weights_ew = available.div(n_available, axis=0).fillna(0)
mask_ew = weights_ew.index >= nav_ew.index[0]
turnover_ew = weights_ew.loc[mask_ew].diff().abs().sum(axis=1).mean() * 12

def calc_stats_with_turnover(nav, ret, name, turnover):
    s = calc_stats(nav, ret, name)
    s['Turnover'] = f"{turnover:.0%}"
    return s

stats = pd.DataFrame([
    calc_stats_with_turnover(nav_regime, ret_regime, 'Regime-Based', turnover_regime),
    calc_stats_with_turnover(nav_ew, ret_ew, 'Equal Weight (BM)', turnover_ew),
]).set_index('Strategy')

print(stats.to_string())
print(f"\nRegime weights:")
for k, v in REGIME_WEIGHTS.items():
    print(f"  {k}: {v:.2f}")

                    CAGR Volatility Sharpe     MDD Final NAV Turnover
Strategy                                                             
Regime-Based       13.9%      15.7%   0.89  -18.2%      4.54     526%
Equal Weight (BM)  12.7%      16.0%   0.80  -22.6%      4.07       4%

Regime weights:
  Bullish: 1.00
  Pullback: 1.00
  Recovery: 0.67
  Correction: 0.33
  Bounce: 0.00
  Bearish: 0.00


In [30]:
# NAV 비교 차트
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['NAV Comparison', 'Drawdown'],
                    row_heights=[0.65, 0.35], vertical_spacing=0.08)

# NAV
fig.add_trace(go.Scatter(
    x=nav_regime.index, y=nav_regime.values,
    name='Regime-Based', line=dict(color='#2ecc71', width=2),
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=nav_ew.index, y=nav_ew.values,
    name='Equal Weight (BM)', line=dict(color='gray', width=1.5, dash='dash'),
), row=1, col=1)

# Drawdown
for nav, name, color, dash in [
    (nav_regime, 'Regime-Based', '#2ecc71', None),
    (nav_ew, 'Equal Weight (BM)', 'gray', 'dash'),
]:
    dd = (nav - nav.cummax()) / nav.cummax()
    fig.add_trace(go.Scatter(
        x=dd.index, y=dd.values,
        name=name, line=dict(color=color, width=1.5, dash=dash),
        showlegend=False, fill='tozeroy', fillcolor=f'rgba({",".join(str(int(color.lstrip("#")[i:i+2], 16)) for i in (0,2,4))},0.1)' if color != 'gray' else 'rgba(128,128,128,0.1)',
    ), row=2, col=1)

fig.update_yaxes(tickformat='.0%', row=2, col=1)
fig.update_layout(
    template='plotly_white',
    height=600, width=900,
    title='CS Regime-Based Strategy vs Equal Weight Benchmark',
)
fig.show()

In [14]:
# 국면별 티커 수 비율 시계열
regime_counts = pd.DataFrame(index=regime_df.index)
for regime_name in ALL_REGIMES:
    regime_counts[regime_name] = (regime_df == regime_name).sum(axis=1)

total = regime_counts.sum(axis=1)
regime_pct = regime_counts.div(total, axis=0)

fig = go.Figure()
for regime_name in ALL_REGIMES:
    color = REGIME_COLORS[regime_name]
    fig.add_trace(go.Scatter(
        x=regime_pct.index,
        y=regime_pct[regime_name],
        name=regime_name,
        stackgroup='one',
        fillcolor=color,
        line=dict(width=0.5, color=color),
    ))

fig.update_layout(
    title='CS Regime Distribution Over Time (% of Universe)',
    yaxis=dict(tickformat='.0%', title='Proportion'),
    template='plotly_white',
    height=400, width=900,
    legend=dict(x=0.01, y=0.99),
)
fig.show()

## 파라미터 최적화

그리드 서치로 최적 파라미터를 탐색합니다:
- **Fast window**: 1, 2, 3, 6개월
- **Slow window**: 9, 10, 11, 12개월
- **약세 최소 비중**: 0, 0.1, 0.2
- **비중 스프레드**: 강세~약세 간 비중 차이 구조

In-sample (2015~2020) / Out-of-sample (2021~) 분할로 오버피팅을 검증합니다.

### IS vs OOS Sharpe 산점도

오버피팅 여부를 시각적으로 확인합니다. 대각선에 가까울수록 IS/OOS 성과가 일관적입니다.

In [17]:
from itertools import product

def build_regime_df(prices, fast_window, slow_window):
    """주어진 윈도우로 국면 DataFrame 생성 (vectorized)."""
    mp = prices.resample('BME').last()
    f_ret = mp.pct_change(fast_window)
    s_ret = mp.pct_change(slow_window)
    rdf = classify_regime_df(s_ret, f_ret)
    return rdf.dropna(how='all'), f_ret, s_ret


def regime_backtest_stats(prices, rdf, regime_weights, tcost=0.000, start='2015-01-01', end=None):
    """백테스트 후 Sharpe 등 통계 반환."""
    nav, _, p_ret = regime_backtest(prices, rdf, regime_weights, tcost=tcost, backtest_start=start)

    if end:
        mask = nav.index < pd.Timestamp(end)
        nav = nav.loc[mask]
        p_ret = p_ret.loc[nav.index]

    if len(nav) < 12:
        return None

    years = (nav.index[-1] - nav.index[0]).days / 365.25
    cagr = (nav.iloc[-1] / nav.iloc[0]) ** (1 / years) - 1
    vol = p_ret.std() * np.sqrt(12)
    sharpe = cagr / vol if vol > 0 else 0
    mdd = ((nav - nav.cummax()) / nav.cummax()).min()

    # 턴오버: 비중 변화 절대합의 연환산
    raw_w = rdf.replace(regime_weights).apply(pd.to_numeric, errors='coerce')
    w_sum = raw_w.sum(axis=1)
    w = raw_w.div(w_sum, axis=0).fillna(0)
    tc = w.diff().abs().sum(axis=1).fillna(0)
    tc = tc.reindex(nav.index)
    turnover = tc.mean() * 12

    return {'cagr': cagr, 'vol': vol, 'sharpe': sharpe, 'mdd': mdd, 'turnover': turnover, 'nav': nav}


# 그리드 서치 파라미터
fast_windows = [1, 2, 3]
slow_windows = [6, 9, 10, 11, 12]
bearish_floors = [0.0, 0.1, 0.2]

# make_weights()는 상단 config 셀에 정의됨

IS_END = '2025-01-01'  # In-sample 끝

print("Running grid search...")
print(f"Combinations: {len(fast_windows) * len(slow_windows) * len(bearish_floors)}")

grid_results = []
regime_cache = {}

for fw, sw, bf in product(fast_windows, slow_windows, bearish_floors):
    if fw >= sw:  # fast는 slow보다 짧아야 함
        continue
    
    # 캐시: 같은 윈도우 조합은 regime_df 재사용
    cache_key = (fw, sw)
    if cache_key not in regime_cache:
        regime_cache[cache_key] = build_regime_df(prices, fw, sw)
    rdf, _, _ = regime_cache[cache_key]
    
    rw = make_weights(bf)
    
    # In-sample
    is_stats = regime_backtest_stats(prices, rdf, rw, start='2015-01-01', end=IS_END)
    # Out-of-sample
    oos_stats = regime_backtest_stats(prices, rdf, rw, start=IS_END)
    
    if is_stats and oos_stats:
        grid_results.append({
            'Fast': fw, 'Slow': sw, 'Bear Floor': bf,
            'Weights': '/'.join(f"{rw[r]:.2f}" for r in ALL_REGIMES),
            'IS Sharpe': is_stats['sharpe'],
            'IS CAGR': is_stats['cagr'],
            'IS MDD': is_stats['mdd'],
            'OOS Sharpe': oos_stats['sharpe'],
            'OOS CAGR': oos_stats['cagr'],
            'OOS MDD': oos_stats['mdd'],
            'Turnover': is_stats['turnover'],
        })

grid_df = pd.DataFrame(grid_results)
print(f"Valid combinations: {len(grid_df)}")

# IS Sharpe 기준 상위 10개
top10 = grid_df.sort_values('IS Sharpe', ascending=False).head(10).copy()
for col in ['IS Sharpe', 'OOS Sharpe']:
    top10[col] = top10[col].map('{:.2f}'.format)
for col in ['IS CAGR', 'OOS CAGR', 'IS MDD', 'OOS MDD']:
    top10[col] = top10[col].map('{:.1%}'.format)
top10['Turnover'] = top10['Turnover'].map('{:.0%}'.format)

print("\n=== Top 10 by In-Sample Sharpe ===")
print(top10.to_string(index=False))

Running grid search...
Combinations: 45
Valid combinations: 45

=== Top 10 by In-Sample Sharpe ===
 Fast  Slow  Bear Floor                       Weights IS Sharpe IS CAGR IS MDD OOS Sharpe OOS CAGR OOS MDD Turnover
    3    11         0.0 1.00/1.00/0.33/0.67/0.00/0.00      0.91   14.4% -18.0%       1.64    23.7%   -5.5%     535%
    2    11         0.0 1.00/1.00/0.33/0.67/0.00/0.00      0.88   14.0% -19.2%       1.58    22.1%   -5.7%     577%
    3    11         0.1 1.00/1.00/0.40/0.70/0.10/0.10      0.88   14.0% -18.7%       1.67    23.2%   -5.4%     436%
    1    11         0.0 1.00/1.00/0.33/0.67/0.00/0.00      0.87   13.8% -18.4%       1.75    25.3%   -5.5%     604%
    1     6         0.0 1.00/1.00/0.33/0.67/0.00/0.00      0.87   13.9% -19.3%       1.62    24.5%   -6.1%     656%
    1     9         0.0 1.00/1.00/0.33/0.67/0.00/0.00      0.87   13.7% -19.9%       1.84    26.7%   -5.0%     615%
    2    11         0.1 1.00/1.00/0.40/0.70/0.10/0.10      0.86   13.7% -19.4%       1.63

### 최적 파라미터 vs 기존 전략 vs BM 비교

IS Sharpe 상위 중 OOS에서도 안정적인 파라미터를 선정하여 비교합니다.

In [16]:
# IS vs OOS Sharpe 산점도
fig = go.Figure()

for bf in bearish_floors:
    subset = grid_df[grid_df['Bear Floor'] == bf]
    fig.add_trace(go.Scatter(
        x=subset['IS Sharpe'], y=subset['OOS Sharpe'],
        mode='markers',
        name=f'Bear Floor={bf}',
        marker=dict(size=10, opacity=0.7),
        text=[f"F{r['Fast']}/S{r['Slow']}" for _, r in subset.iterrows()],
        hovertemplate='%{text}<br>IS: %{x:.2f}<br>OOS: %{y:.2f}<extra></extra>',
    ))

# 대각선
sharpe_range = [min(grid_df['IS Sharpe'].min(), grid_df['OOS Sharpe'].min()),
                max(grid_df['IS Sharpe'].max(), grid_df['OOS Sharpe'].max())]
fig.add_trace(go.Scatter(
    x=sharpe_range, y=sharpe_range,
    mode='lines', line=dict(dash='dash', color='gray'),
    showlegend=False,
))

fig.update_layout(
    title='In-Sample vs Out-of-Sample Sharpe Ratio',
    xaxis_title='In-Sample Sharpe (2015~2020)',
    yaxis_title='Out-of-Sample Sharpe (2021~)',
    template='plotly_white',
    height=500, width=600,
)
fig.show()